### Advanced Multi-Source Data Pipeline: CoinGecko API + Web Scraping → Data Cleaning → Exploratory Data Analysis

In [ ]:
# Step 1: Import libraries required 

import requests
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

In [ ]:
# STEP 2: COINGECKO API DATA COLLECTION
# CoinGecko API endpoint
url = "https://api.coingecko.com/api/v3/coins/markets"

# Parameters for the API request
params = {
    "vs_currency": "usd",              # Prices will be in USD
    "order": "market_cap_desc",        # Highest market cap first
    "per_page": 250,                   # Collect up to 250 cryptocurrencies
    "page": 1,
    "sparkline": "false"               # We don't need sparkline data
}

# Send request to CoinGecko
response = requests.get(url, params=params, timeout=30)

# Check whether the request was successful
print("Status Code:", response.status_code)

# Convert API response into Python data
crypto_data = response.json()

# Check how many cryptocurrency records were received
print("Number of records collected:", len(crypto_data))

In [ ]:
# Convert the API response into a Pandas DataFrame
crypto_df = pd.DataFrame(crypto_data)

# Display the first 5 records
crypto_df.head()

In [ ]:
# Display all column names returned by the API
print("Columns in CoinGecko dataset:")

for column in crypto_df.columns:
    print(column)

In [ ]:
# Number of rows and columns
print("Dataset Shape:", crypto_df.shape)

In [ ]:
# Display data Information
crypto_df.info()

In [ ]:
# Missing values in every column
crypto_df.isnull().sum()

In [ ]:
# Check duplicate cryptocurrency IDs
print("Duplicate IDs:", crypto_df["id"].duplicated().sum())

# Check duplicate rows using only the normal scalar columns.
# This avoids dictionary/list columns that cannot be hashed.
hashable_columns = [
    col for col in crypto_df.columns
    if not crypto_df[col].map(lambda x: isinstance(x, (dict, list, set))).any()
]

print("Columns used for duplicate-row check:", len(hashable_columns))
print("Duplicate rows:",crypto_df[hashable_columns].duplicated().sum())

In [ ]:
# Find columns containing dictionary, list, or set values
complex_columns = []

for column in crypto_df.columns:
    if crypto_df[column].map(lambda x: isinstance(x, (dict, list, set))).any():
        complex_columns.append(column)

print("Columns containing complex values:")
print(complex_columns)

In [ ]:
# STEP 3: INITIAL CLEANING OF COINGECKO DATA

# 1. Remove the 'roi' column
if "roi" in crypto_df.columns:
    crypto_df.drop(columns=["roi"], inplace=True)

# 2. Check for duplicate cryptocurrency IDs
print("Duplicate IDs before removal:", crypto_df["id"].duplicated().sum())

# 3. Permanently remove duplicate cryptocurrencies
crypto_df.drop_duplicates(subset="id", keep="first", inplace=True)

print("Shape after duplicate removal:", crypto_df.shape)

# 4. Check missing values
print("\nMissing values:")
missing_values = crypto_df.isnull().sum()
print(missing_values[missing_values > 0])

# 5. Check data types
print("\nData types:")
print(crypto_df.dtypes)

In [ ]:
# STEP 3.1: NUMERIC DATA TYPE CONVERSION

# Columns that should contain numeric values
numeric_columns = [
    "current_price",
    "market_cap",
    "total_volume",
    "high_24h",
    "low_24h",
    "price_change_24h",
    "price_change_percentage_24h",
    "market_cap_change_24h",
    "market_cap_change_percentage_24h",
    "circulating_supply",
    "total_supply",
    "max_supply"
]

# Convert only the columns that actually exist in the dataset
available_numeric_columns = [
    column for column in numeric_columns
    if column in crypto_df.columns
]

for column in available_numeric_columns:
    crypto_df[column] = pd.to_numeric(crypto_df[column],errors="coerce")

# Display the converted data types
print("Numeric columns converted successfully:\n")

for column in available_numeric_columns:
    print(f"{column}: {crypto_df[column].dtype}")

In [ ]:
# STEP 3.2: DATETIME CONVERSION

# Convert into proper datetime format
if "last_updated" in crypto_df.columns:
    crypto_df["last_updated"] = pd.to_datetime(crypto_df["last_updated"],errors="coerce",utc=True)

# Check the result
print("last_updated data type:", crypto_df["last_updated"].dtype)

# Display a few values
crypto_df[["id", "last_updated"]].head()

In [ ]:
# STEP 3.3: MISSING VALUE ANALYSIS

# Count missing values in each column
missing_summary = crypto_df.isnull().sum()

# Keep only columns that actually contain missing values
missing_summary = missing_summary[missing_summary > 0]

print("Missing values before cleaning:\n")
print(missing_summary)

print("\nTotal missing values:", crypto_df.isnull().sum().sum())

In [ ]:
# STEP 3.4 FINAL FIX: HANDLE REMAINING MISSING VALUES

# Fill the 1 missing price-change value with its median
crypto_df["price_change_24h"] = crypto_df["price_change_24h"].fillna(
    crypto_df["price_change_24h"].median()
)

# Fill the 2 missing market-cap-change values with their median
crypto_df["market_cap_change_24h"] = crypto_df["market_cap_change_24h"].fillna(crypto_df["market_cap_change_24h"].median())

# Verify the result
remaining_missing = crypto_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print("Remaining missing values:")
print(remaining_missing)
print("\nTotal remaining missing values:",crypto_df.isnull().sum().sum())

In [ ]:
# STEP 3.5: VALIDATE AND REMOVE INVALID PRICES

# Count invalid prices before removal
invalid_price_count = ((crypto_df["current_price"] <= 0) |(crypto_df["current_price"].isnull())).sum()
print("Invalid current_price records:", invalid_price_count)

# Remove records where current_price is zero, negative, or missing
crypto_df = crypto_df[(crypto_df["current_price"] > 0) &(crypto_df["current_price"].notna())].reset_index(drop=True)
print("Shape after removing invalid prices:", crypto_df.shape)

# Verify that no invalid prices remain
print("Invalid prices remaining:",((crypto_df["current_price"] <= 0) |(crypto_df["current_price"].isnull())).sum())

In [ ]:
# VERIFY MISSING VALUES AFTER CLEANING
print("Remaining missing values:")
remaining_missing = crypto_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(remaining_missing)
print("\nTotal remaining missing values:",crypto_df.isnull().sum().sum())

In [ ]:
# STEP 3.6: OUTLIER DETECTION USING IQR

# Numerical columns where outliers are meaningful for analysis
outlier_columns = [
    "current_price",
    "market_cap",
    "total_volume"
]

# Store IQR outlier counts
iqr_outlier_summary = {}

for column in outlier_columns:

    # Calculate Q1 and Q3
    Q1 = crypto_df[column].quantile(0.25)
    Q3 = crypto_df[column].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Calculate lower and upper limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = ((crypto_df[column] < lower_limit) |(crypto_df[column] > upper_limit))

    # Store number of outliers
    iqr_outlier_summary[column] = outliers.sum()

    print(f"\n{column}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Limit:", lower_limit)
    print("Upper Limit:", upper_limit)
    print("Number of IQR outliers:", outliers.sum())

# Display summary
print("\n----------------------------------------------------------")
print("IQR OUTLIER SUMMARY")
print("------------------------------------------------------------")

print(pd.Series(iqr_outlier_summary))

In [ ]:
# STEP 3.7: OUTLIER DETECTION USING ISOLATION FOREST

# Select numerical features for outlier detection
isolation_features = [
    "current_price",
    "market_cap",
    "total_volume"
]

# Create a temporary dataset containing only the required features
isolation_data = crypto_df[isolation_features].copy()

# Create and train the Isolation Forest model
isolation_model = IsolationForest(contamination=0.05,random_state=42)

# Predict outliers
#  1  = normal observation
# -1  = outlier
crypto_df["isolation_forest_flag"] = isolation_model.fit_predict(isolation_data)

# Convert the prediction into an easy-to-understand label
crypto_df["isolation_outlier"] = np.where(crypto_df["isolation_forest_flag"] == -1,"Outlier","Normal")

# Count the results
print("Isolation Forest results:")
print(crypto_df["isolation_outlier"].value_counts())

In [ ]:
# STEP 3.8: INSPECT ISOLATION FOREST OUTLIERS

# Display the cryptocurrencies identified as potential outliers
outlier_crypto = crypto_df[
    crypto_df["isolation_outlier"] == "Outlier"
][
    [
        "id",
        "symbol",
        "current_price",
        "market_cap",
        "total_volume",
        "price_change_percentage_24h",
        "isolation_outlier"
    ]
]

print("Potential Isolation Forest Outliers:")
display(outlier_crypto)

In [ ]:
# STEP 3.9: FINALIZE OUTLIER HANDLING

crypto_df["outlier_status"] = crypto_df["isolation_outlier"]

# Remove the temporary Isolation Forest prediction column
crypto_df.drop(columns=["isolation_forest_flag"], inplace=True)
print("Outlier handling completed.")
print("\nOutlier status:")
print(crypto_df["outlier_status"].value_counts())
print("\nCurrent dataset shape:", crypto_df.shape)

In [ ]:
# STEP 4: CRYPTO FEATURE ENGINEERING

# 4.1 MARKET CAP CATEGORY
crypto_df["market_cap_category"] = np.select([
        crypto_df["market_cap"] >= 10_000_000_000,
        crypto_df["market_cap"] >= 1_000_000_000
    ],
    ["Large Cap",
      "Mid Cap"
    ],
    default="Small Cap"
)

# 4.2 VOLATILITY SCORE
crypto_df["volatility_score"] = (crypto_df["price_change_percentage_24h"].abs())

# 4.3 SUPPLY RATIO
crypto_df["supply_ratio"] = np.where((crypto_df["total_supply"].notna()) &(crypto_df["total_supply"] > 0),
    crypto_df["circulating_supply"] / crypto_df["total_supply"],np.nan)


# CHECK THE NEW FEATURES
print("Feature engineering completed successfully!")
print("\nNew columns:")
print([
    "market_cap_category",
    "volatility_score",
    "supply_ratio"
])
print("\nMarket Cap Category:")
print(crypto_df["market_cap_category"].value_counts())
print("\nSample of engineered features:")
display(crypto_df[
        [
            "id",
            "symbol",
            "market_cap",
            "market_cap_category",
            "price_change_percentage_24h",
            "volatility_score",
            "supply_ratio"
        ]
    ].head(10)
)

In [ ]:
# STEP 5.1: BOOKS TO SCRAPE - DATA COLLECTION

base_url = "https://books.toscrape.com/"
page_url = base_url

books_data = []

while page_url:

    response = requests.get(page_url, timeout=30)

    if response.status_code != 200:
        print("Failed to access:", page_url)
        break

    soup = BeautifulSoup(response.text, "html.parser")

    # Find all books on the current page
    books = soup.select("article.product_pod")

    for book in books:

        # Book title
        title = book.select_one("h3 a")["title"].strip()

        # Price
        price = book.select_one("p.price_color").get_text(strip=True)

        # Rating
        rating_element = book.select_one("p.star-rating")
        rating = rating_element["class"][1] if rating_element else None

        # Availability
        availability_element = book.select_one("p.instock.availability")
        availability = (availability_element.get_text(" ", strip=True) if availability_element else None)

        # Product URL
        product_link = book.select_one("h3 a")["href"]
        product_url = requests.compat.urljoin(page_url,product_link)

        # Category
        category = None

        books_data.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "category": category,
            "product_url": product_url
        })

    # Find the next page
    next_button = soup.select_one("li.next a")

    if next_button:
        page_url = requests.compat.urljoin(page_url,next_button["href"])
    else:
        page_url = None

# Convert collected records into DataFrame
books_df = pd.DataFrame(books_data)

print("Books data collection completed successfully!")
print("Number of books collected:", len(books_df))
print("Dataset shape:", books_df.shape)

display(books_df.head())

In [ ]:
# STEP 5.2: COLLECT BOOK CATEGORIES WITH PROGRESS

# Get the homepage
response = requests.get(base_url, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

# Find all 50 category links
category_links = soup.select("ul.nav-list ul li a")
print("Number of category links found:", len(category_links))

# Dictionary:
# product_url -> category
category_map = {}

# Process each category
for category_number, link in enumerate(category_links, start=1):
    category_name = link.get_text(strip=True)
    category_url = requests.compat.urljoin(base_url,link.get("href"))
    page_url = category_url
    category_book_count = 0

    # Process all pages inside this category
    while page_url:
        response = requests.get(page_url, timeout=30)

        if response.status_code != 200:
            print("Failed to access:", page_url)
            break

        soup = BeautifulSoup(response.text, "html.parser")

        # Find books on this category page
        books = soup.select("article.product_pod")

        for book in books:
            product_link = book.select_one("h3 a")["href"]
            product_url = requests.compat.urljoin(page_url,product_link)
            category_map[product_url] = category_name
            category_book_count += 1

        # Find next page
        next_button = soup.select_one("li.next a")

        if next_button:
            page_url = requests.compat.urljoin(page_url,next_button["href"])
        else:
            page_url = None

    # Show progress after every category
    print(
        f"Category {category_number}/{len(category_links)} "
        f"completed: {category_name} | "
        f"Books found: {category_book_count} | "
        f"Total mapped: {len(category_map)}"
    )

print("\n============================================================")
print("CATEGORY COLLECTION COMPLETED")
print("============================================================")
print("Total product-category mappings:", len(category_map))

In [ ]:
# STEP 5.3: ADD CATEGORIES TO BOOKS DATAFRAME

# Match each book's product URL with its category
books_df["category"] = books_df["product_url"].map(category_map)
print("Categories added successfully!")
print("\nMissing categories:")
print(books_df["category"].isnull().sum())
print("\nDataset shape:", books_df.shape)
print("\nCategory distribution:")
print(books_df["category"].value_counts())
print("\nSample data:")
display(
    books_df[
        [
            "title",
            "price",
            "rating",
            "availability",
            "category"
        ]
    ].head(10)
)

In [ ]:
# STEP 5.4: INITIAL INSPECTION OF BOOKS DATA

print("Dataset Shape:", books_df.shape)

print("\nColumn Names:")
print(books_df.columns.tolist())

print("\nData Types:")
print(books_df.dtypes)

print("\nDuplicate Records:")
print(books_df.duplicated().sum())

print("\nDuplicate Titles:")
print(books_df["title"].duplicated().sum())

print("\nMissing Values:")
print(books_df.isnull().sum())

print("\nSample Records:")
display(books_df.sample(10, random_state=42))

In [ ]:
# STEP 6.1: REMOVE DUPLICATE BOOK TITLES

print("Duplicate titles before removal:",books_df["title"].duplicated().sum())

# Keep the first occurrence and permanently remove duplicates
books_df.drop_duplicates(subset="title",keep="first",inplace=True)

# Reset index after removing duplicate
books_df.reset_index(drop=True, inplace=True)

print("Duplicate titles after removal:",
      books_df["title"].duplicated().sum())

print("Dataset shape after duplicate removal:",
      books_df.shape)

In [ ]:
# STEP 6.2: CLEAN BOOK PRICE

# Remove the currency symbol and convert price to numeric
books_df["price"] = (books_df["price"].str.replace("Â£", "", regex=False).str.strip())
books_df["price"] = pd.to_numeric(books_df["price"],errors="coerce")
print("Price conversion completed!")
print("\nPrice data type:")
print(books_df["price"].dtype)
print("\nMissing prices after conversion:")
print(books_df["price"].isnull().sum())
print("\nPrice summary:")
print(books_df["price"].describe())

In [ ]:
# STEP 6.3: CONVERT BOOK RATINGS TO NUMERIC VALUES

# Mapping text ratings to numerical ratings
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

books_df["rating"] = books_df["rating"].map(rating_mapping)
print("Rating conversion completed!")
print("\nRating data type:")
print(books_df["rating"].dtype)
print("\nMissing ratings:")
print(books_df["rating"].isnull().sum())
print("\nRating distribution:")
print(books_df["rating"].value_counts().sort_index())

In [ ]:
# STEP 6.4: CLEAN BOOK AVAILABILITY

# Remove the incorrect stock_quantity column if it exists
if "stock_quantity" in books_df.columns:
    books_df.drop(columns=["stock_quantity"], inplace=True)

# Create stock status from the availability information
books_df["stock_status"] = np.where(books_df["availability"].str.strip().str.lower() == "in stock","In Stock","Out of Stock")

print("Stock status created successfully!")
print("\nStock status distribution:")
print(books_df["stock_status"].value_counts())
print("\nMissing stock status:")
print(books_df["stock_status"].isnull().sum())

print("\nSample:")
display(
    books_df[
        ["title", "availability", "stock_status"]
    ].head(10)
)

In [ ]:
# STEP 6.5: CLEAN BOOK TITLES

# Remove unnecessary spaces from the beginning and end
books_df["title"] = books_df["title"].str.strip()

# Replace multiple spaces with a single space
books_df["title"] = books_df["title"].str.replace(r"\s+"," ",regex=True)
print("Title cleaning completed!")
print("\nMissing titles:")
print(books_df["title"].isnull().sum())
print("\nSample cleaned titles:")
display(books_df[["title"]].head(10))

In [ ]:
# STEP 6.6: VALIDATE CLEANED DATA TYPES

print("Data types after cleaning:\n")
print(books_df.dtypes)
print("\nMissing values:")
print(books_df.isnull().sum())
print("\nPrice type:", books_df["price"].dtype)
print("Rating type:", books_df["rating"].dtype)
print("Stock Status type:", books_df["stock_status"].dtype)
print("\nPrice range:")
print("Minimum price:", books_df["price"].min())
print("Maximum price:", books_df["price"].max())
print("\nRating range:")
print("Minimum rating:", books_df["rating"].min())
print("Maximum rating:", books_df["rating"].max())

In [ ]:
# STEP 7.1: CREATE PRICE RANGE CATEGORY

# Categorize books based on their price
books_df["price_range_category"] = np.select([books_df["price"] < 20,books_df["price"] <= 40],
    ["Low",
     "Medium"
    ],
    default="High"
)

print("Price range category created successfully!")
print("\nPrice range distribution:")
print(books_df["price_range_category"].value_counts())
print("\nSample:")
display(books_df[["title", "price", "price_range_category"]
    ].head(10)
)

In [ ]:
# STEP 7.2: CREATE RATING GROUP

# Group books according to their rating
books_df["rating_group"] = np.select([books_df["rating"] <= 2,books_df["rating"] == 3],
    ["Low Rating",
    "Average Rating"
    ],
    default="High Rating"
)

print("Rating group created successfully!")
print("\nRating group distribution:")
print(books_df["rating_group"].value_counts())
print("\nSample:")
display(books_df[["title", "rating", "rating_group"]
    ].head(10)
)

In [ ]:
# STEP 7.3: VALIDATE BOOK FEATURE ENGINEERING

print("Feature engineering validation:")
print("\nPrice Range Categories:")
print(books_df["price_range_category"].value_counts())
print("\nRating Groups:")
print(books_df["rating_group"].value_counts())
print("\nStock Status:")
print(books_df["stock_status"].value_counts())
print("\nMissing values in engineered features:")
print(
    books_df[
        [
            "price_range_category",
            "rating_group",
            "stock_status"
        ]
    ].isnull().sum()
)

In [ ]:
# STEP 7.4: FINAL BOOKS DATA VALIDATION

print("============================================================")
print("FINAL BOOKS DATA VALIDATION")
print("============================================================")

# Check dataset shape
print("\nDataset Shape:")
print(books_df.shape)

# Check duplicate titles
print("\nDuplicate Titles:")
print(books_df["title"].duplicated().sum())

# Check missing values
print("\nMissing Values:")
print(books_df.isnull().sum())

# Check invalid prices
print("\nInvalid Prices (<= 0):")
print((books_df["price"] <= 0).sum())

# Check invalid ratings
print("\nInvalid Ratings:")
print((~books_df["rating"].isin([1, 2, 3, 4, 5])).sum())

# Check engineered features
print("\nEngineered Feature Missing Values:")
print(
    books_df[
        [
            "price_range_category",
            "rating_group",
            "stock_status"
        ]
    ].isnull().sum()
)

print("\n============================================================")
print("FINAL VALIDATION COMPLETED")
print("============================================================")

In [ ]:
# STEP 8.1: BOOK PRICE DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.histplot(
    data=books_df,
    x="price",
    bins=20,
    kde=True
)
plt.title("Distribution of Book Prices")
plt.xlabel("Price (£)")
plt.ylabel("Number of Books")
plt.show()

In [ ]:
# STEP 8.2: BOOK RATING DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.countplot(
    data=books_df,
    x="rating"
)
plt.title("Distribution of Book Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Books")
plt.show()

In [ ]:
# STEP 8.3: BOOK PRICE BY RATING

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=books_df,
    x="rating",
    y="price"
)
plt.title("Book Price Distribution by Rating")
plt.xlabel("Rating")
plt.ylabel("Price (£)")
plt.show()

In [ ]:
# STEP 8.4: BOOK CATEGORY DISTRIBUTION

# Count the number of books in each category
category_counts = books_df["category"].value_counts()
plt.figure(figsize=(12, 8))
sns.barplot(
    x=category_counts.values,
    y=category_counts.index
)
plt.title("Number of Books by Category")
plt.xlabel("Number of Books")
plt.ylabel("Book Category")
plt.show()

In [ ]:
# STEP 8.5: PRICE RANGE CATEGORY DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.countplot(
    data=books_df,
    x="price_range_category"
)
plt.title("Distribution of Books by Price Range")
plt.xlabel("Price Range Category")
plt.ylabel("Number of Books")
plt.show()

In [ ]:
# STEP 8.6: RATING GROUP DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.countplot(
    data=books_df,
    x="rating_group"
)
plt.title("Distribution of Books by Rating Group")
plt.xlabel("Rating Group")
plt.ylabel("Number of Books")
plt.show()

In [ ]:
# STEP 8.7: AVERAGE BOOK PRICE BY RATING

# Calculate average price for each rating
average_price_by_rating = (
    books_df
    .groupby("rating")["price"]
    .mean()
    .reset_index()
)
print("Average book price by rating:")
display(average_price_by_rating)

# Plot the result
plt.figure(figsize=(10, 6))
sns.barplot(
    data=average_price_by_rating,
    x="rating",
    y="price"
)

plt.title("Average Book Price by Rating")
plt.xlabel("Rating")
plt.ylabel("Average Price (£)")
plt.show()

In [ ]:
# STEP 8.8: TOP 10 CATEGORIES BY AVERAGE PRICE

# Calculate average price for each category
category_avg_price = (books_df.groupby("category")["price"].mean().sort_values(ascending=False).head(10).reset_index())
print("Top 10 categories by average book price:")
display(category_avg_price)

# Plot the result
plt.figure(figsize=(12, 7))
sns.barplot(
    data=category_avg_price,
    x="price",
    y="category"
)

plt.title("Top 10 Categories by Average Book Price")
plt.xlabel("Average Price (£)")
plt.ylabel("Book Category")
plt.show()

In [ ]:
# STEP 8.9: RATING VS PRICE RELATIONSHIP

plt.figure(figsize=(10, 6))
sns.regplot(
    data=books_df,
    x="rating",
    y="price",
    scatter_kws={"alpha": 0.6},
    line_kws={"linewidth": 2}
)
plt.title("Relationship Between Book Rating and Price")
plt.xlabel("Rating")
plt.ylabel("Price (£)")
plt.show()

# Calculate correlation
correlation = books_df["rating"].corr(books_df["price"])
print("Correlation between Rating and Price:", round(correlation, 4))

In [ ]:
# STEP 8.10: AVERAGE BOOK RATING BY CATEGORY

# Calculate average rating for each book category
category_avg_rating = (books_df.groupby("category")["rating"].mean().sort_values(ascending=False).head(10).reset_index())

print("Top 10 categories by average book rating:")
display(category_avg_rating)

# Plot the result
plt.figure(figsize=(12, 7))
sns.barplot(
    data=category_avg_rating,
    x="rating",
    y="category"
)

plt.title("Top 10 Categories by Average Book Rating")
plt.xlabel("Average Rating")
plt.ylabel("Book Category")
plt.show()

In [ ]:
# STEP 8.11: PRICE RANGE VS RATING GROUP

# Create a frequency table showing rating groups
price_rating_table = pd.crosstab(books_df["price_range_category"],books_df["rating_group"])
print("Price Range vs Rating Group:")
display(price_rating_table)

# Plot the relationship as a heatmap
plt.figure(figsize=(10, 6))

sns.heatmap(
    price_rating_table,
    annot=True,
    fmt="d"
)

plt.title("Price Range vs Rating Group")
plt.xlabel("Rating Group")
plt.ylabel("Price Range Category")
plt.show()

In [ ]:
# STEP 8.12: BOOK PRICE OUTLIER ANALYSIS

# Calculate Q1, Q3 and IQR
Q1 = books_df["price"].quantile(0.25)
Q3 = books_df["price"].quantile(0.75)

IQR = Q3 - Q1

# Calculate the lower and upper limits
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Identify price outliers
books_df["price_outlier"] = np.where(
    (books_df["price"] < lower_limit) |
    (books_df["price"] > upper_limit),
    "Outlier",
    "Normal"
)

print("Book Price Outlier Analysis")
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)
print("\nOutlier distribution:")
print(books_df["price_outlier"].value_counts())
print("\nPotential price outliers:")
display(books_df[books_df["price_outlier"] == "Outlier"][["title", "price", "rating", "category", "price_outlier"]].sort_values("price", ascending=False))

In [ ]:
# STEP 8.13: SPEARMAN CORRELATION - RATING VS PRICE

# Calculate Spearman correlation and p-value
spearman_corr, spearman_p = stats.spearmanr(books_df["price"],books_df["rating"])
print("Spearman Correlation:", round(spearman_corr, 4))
print("P-value:", round(spearman_p, 6))

In [ ]:
# STEP 8.14: INTERACTIVE BOOK PRICE VS RATING

fig = px.scatter(
    books_df,
    x="rating",
    y="price",
    color="rating_group",
    hover_data=[
        "title",
        "category",
        "price_range_category",
        "stock_status"
    ],
    title="Interactive Book Price vs Rating",
    labels={
        "rating": "Book Rating",
        "price": "Price (£)",
        "rating_group": "Rating Group"
    }
)

fig.show()

In [ ]:
# STEP 9.1: CRYPTOCURRENCY PRICE DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.histplot(
    data=crypto_df,
    x="current_price",
    bins=30,
    kde=True
)
plt.title("Distribution of Cryptocurrency Prices")
plt.xlabel("Current Price (USD)")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.2: CRYPTOCURRENCY MARKET CAP DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.histplot(
    data=crypto_df,
    x="market_cap",
    bins=30,
    kde=True
)
plt.title("Distribution of Cryptocurrency Market Capitalization")
plt.xlabel("Market Capitalization (USD)")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.3: CRYPTOCURRENCY TRADING VOLUME DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.histplot(
    data=crypto_df,
    x="total_volume",
    bins=30,
    kde=True
)
plt.title("Distribution of Cryptocurrency Trading Volume")
plt.xlabel("24-Hour Trading Volume (USD)")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.4: CRYPTOCURRENCY VOLATILITY DISTRIBUTION

plt.figure(figsize=(10, 6))
sns.histplot(
    data=crypto_df,
    x="volatility_score",
    bins=30,
    kde=True
)
plt.title("Distribution of Cryptocurrency Volatility")
plt.xlabel("Volatility Score (Absolute 24-Hour Price Change %)")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.5: MARKET CAP CATEGORY DISTRIBUTION

# Count cryptocurrencies in each market cap category
market_cap_counts = crypto_df["market_cap_category"].value_counts()
print("Market Cap Category Distribution:")
display(market_cap_counts)

# Plot the distribution
plt.figure(figsize=(10, 6))
sns.countplot(
    data=crypto_df,
    x="market_cap_category"
)
plt.title("Distribution of Cryptocurrencies by Market Cap Category")
plt.xlabel("Market Cap Category")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.6: CRYPTOCURRENCY PRICE VS MARKET CAP

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=crypto_df,
    x="market_cap",
    y="current_price",
    hue="market_cap_category",
    size="volatility_score",
    sizes=(20, 200),
    alpha=0.7
)

plt.title("Cryptocurrency Price vs Market Capitalization")
plt.xlabel("Market Capitalization (USD)")
plt.ylabel("Current Price (USD)")
plt.show()

In [ ]:
# STEP 9.7: CRYPTOCURRENCY PRICE VS TRADING VOLUME

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=crypto_df,
    x="total_volume",
    y="current_price",
    hue="market_cap_category",
    size="volatility_score",
    sizes=(20, 200),
    alpha=0.7
)

plt.title("Cryptocurrency Price vs Trading Volume")
plt.xlabel("24-Hour Trading Volume (USD)")
plt.ylabel("Current Price (USD)")
plt.show()

In [ ]:
# STEP 9.8: CRYPTOCURRENCY CORRELATION HEATMAP

# Select important numerical variables for correlation analysis
crypto_correlation = crypto_df[
    [
        "current_price",
        "market_cap",
        "total_volume",
        "price_change_percentage_24h",
        "volatility_score",
        "supply_ratio"
    ]
].corr()

print("Cryptocurrency Correlation Matrix:")
display(crypto_correlation.round(3))

# Plot the correlation matrix
plt.figure(figsize=(10, 7))
sns.heatmap(
    crypto_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Cryptocurrency Correlation Heatmap")
plt.show()

In [ ]:
# STEP 9.9: MARKET CAP CATEGORY VS PRICE CHANGE %

# Calculate average 24-hour price change for each market cap category
market_cap_change = (
    crypto_df
    .groupby("market_cap_category")["price_change_percentage_24h"]
    .mean()
    .reset_index()
)

print("Average 24-Hour Price Change by Market Cap Category:")
display(market_cap_change)

# Plot the result
plt.figure(figsize=(10, 6))
sns.barplot(
    data=market_cap_change,
    x="market_cap_category",
    y="price_change_percentage_24h"
)

plt.title("Average 24-Hour Price Change by Market Cap Category")
plt.xlabel("Market Cap Category")
plt.ylabel("Average 24-Hour Price Change (%)")
plt.axhline(
    0,
    linewidth=1)
plt.show()

In [ ]:
# STEP 9.10: VOLATILITY BY MARKET CAP CATEGORY

# Calculate average volatility for each market cap category
volatility_by_category = (crypto_df.groupby("market_cap_category")["volatility_score"].mean().sort_values(ascending=False).reset_index())

print("Average Volatility by Market Cap Category:")
display(volatility_by_category)

# Plot the result
plt.figure(figsize=(10, 6))

sns.barplot(
    data=volatility_by_category,
    x="market_cap_category",
    y="volatility_score"
)

plt.title("Average Cryptocurrency Volatility by Market Cap Category")
plt.xlabel("Market Cap Category")
plt.ylabel("Average Volatility Score (%)")
plt.show()

In [ ]:
# STEP 9.11: TOP 10 MOST VOLATILE CRYPTOCURRENCIES

# Select the 10 cryptocurrencies with the highest volatility
top_10_volatile = (
    crypto_df[
        [
            "name",
            "symbol",
            "current_price",
            "price_change_percentage_24h",
            "volatility_score",
            "market_cap_category"
        ]
    ]
    .sort_values(
        "volatility_score",
        ascending=False
    )
    .head(10)
)

print("Top 10 Most Volatile Cryptocurrencies:")
display(top_10_volatile)

# Plot the result
plt.figure(figsize=(12, 7))
sns.barplot(
    data=top_10_volatile,
    x="volatility_score",
    y="name"
)

plt.title("Top 10 Most Volatile Cryptocurrencies")
plt.xlabel("Volatility Score (%)")
plt.ylabel("Cryptocurrency")
plt.show()

In [ ]:
# STEP 9.12: CRYPTOCURRENCY SUPPLY RATIO DISTRIBUTION

# Keep only valid supply ratio values for analysis
valid_supply_ratio = crypto_df[crypto_df["supply_ratio"].notna()]
print("Valid Supply Ratio Records:",
      len(valid_supply_ratio))

print("\nSupply Ratio Summary:")
display(valid_supply_ratio["supply_ratio"].describe())

# Plot the distribution
plt.figure(figsize=(10, 6))
sns.histplot(
    data=valid_supply_ratio,
    x="supply_ratio",
    bins=30,
    kde=True
)

plt.title("Distribution of Cryptocurrency Supply Ratio")
plt.xlabel("Supply Ratio")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.13: SUPPLY RATIO BY MARKET CAP CATEGORY

supply_by_category = (crypto_df.dropna(subset=["supply_ratio"]).groupby("market_cap_category")["supply_ratio"].mean().sort_values(ascending=False).reset_index())

print("Average Supply Ratio by Market Cap Category:")
display(supply_by_category)

# Plot average supply ratio
plt.figure(figsize=(10, 6))
sns.barplot(
    data=supply_by_category,
    x="market_cap_category",
    y="supply_ratio"
)
plt.title("Average Supply Ratio by Market Cap Category")
plt.xlabel("Market Cap Category")
plt.ylabel("Average Supply Ratio")
plt.show()

In [ ]:
# STEP 9.14: TOP 10 CRYPTOCURRENCIES BY MARKET CAPITALIZATION

top_10_market_cap = (
    crypto_df
    .nlargest(10, "market_cap")
    [
        [
            "name",
            "symbol",
            "current_price",
            "market_cap",
            "total_volume",
            "price_change_percentage_24h",
            "market_cap_category"
        ]
    ]
)

print("Top 10 Cryptocurrencies by Market Capitalization:")
display(top_10_market_cap)

In [ ]:
# STEP 9.15: INTERACTIVE CRYPTOCURRENCY MARKET CAP ANALYSIS

fig = px.scatter(
    crypto_df,
    x="market_cap",
    y="current_price",
    color="market_cap_category",
    size="volatility_score",
    hover_data=[
        "name",
        "symbol",
        "total_volume",
        "price_change_percentage_24h",
        "volatility_score"
    ],
    title="Interactive Cryptocurrency Market Cap vs Price",
    labels={
        "market_cap": "Market Capitalization (USD)",
        "current_price": "Current Price (USD)",
        "market_cap_category": "Market Cap Category",
        "volatility_score": "Volatility Score"
    }
)

fig.show()

In [ ]:
# STEP 9.16: CRYPTOCURRENCY OUTLIER ANALYSIS

print("Cryptocurrency Outlier Distribution:")
display(crypto_df["outlier_status"].value_counts())

# Plot outlier distribution
plt.figure(figsize=(10, 6))
sns.countplot(
    data=crypto_df,
    x="outlier_status"
)

plt.title("Cryptocurrency Outlier Distribution")
plt.xlabel("Outlier Status")
plt.ylabel("Number of Cryptocurrencies")
plt.show()

In [ ]:
# STEP 9.17: KRUSKAL-WALLIS TEST
# Volatility Across Market Cap Categories

# Separate volatility values by market cap category
large_cap = crypto_df[crypto_df["market_cap_category"] == "Large Cap"]["volatility_score"].dropna()
mid_cap = crypto_df[crypto_df["market_cap_category"] == "Mid Cap"]["volatility_score"].dropna()
small_cap = crypto_df[crypto_df["market_cap_category"] == "Small Cap"]["volatility_score"].dropna()

# Perform Kruskal-Wallis statistical test
kruskal_stat, kruskal_p = stats.kruskal(large_cap,mid_cap,small_cap)

print("Kruskal-Wallis Test: Volatility Across Market Cap Categories")
print("------------------------------------------------------------")
print("Test Statistic:", round(kruskal_stat, 4))
print("P-value:", round(kruskal_p, 6))

# Interpret the result
if kruskal_p < 0.05:
    print("\nResult: Statistically significant difference in volatility.")
else:
    print("\nResult: No statistically significant difference in volatility.")

In [ ]:
# STEP 9.18: INTERACTIVE CRYPTOCURRENCY VOLATILITY ANALYSIS

fig = px.scatter(
    crypto_df,
    x="market_cap_category",
    y="volatility_score",
    color="outlier_status",
    size="market_cap",
    hover_data=[
        "name",
        "symbol",
        "current_price",
        "market_cap",
        "price_change_percentage_24h"
    ],
    title="Interactive Cryptocurrency Volatility by Market Cap Category",
    labels={
        "market_cap_category": "Market Cap Category",
        "volatility_score": "Volatility Score (%)",
        "outlier_status": "Outlier Status"
    }
)

fig.show()

In [ ]:
# STEP 10.1: FINAL CRYPTOCURRENCY DATA VALIDATION

print("============================================================")
print("FINAL CRYPTOCURRENCY DATA VALIDATION")
print("============================================================")

# Dataset shape
print("\n1. Dataset Shape:")
print(crypto_df.shape)

# Duplicate IDs
print("\n2. Duplicate IDs:")
print(crypto_df["id"].duplicated().sum())

# Missing values
print("\n3. Missing Values:")
print(crypto_df.isnull().sum()[crypto_df.isnull().sum() > 0])

# Invalid prices
print("\n4. Invalid Prices:")
print(((crypto_df["current_price"] <= 0) |(crypto_df["current_price"].isna())).sum())

# Rating of market cap categories
print("\n5. Market Cap Categories:")
print(crypto_df["market_cap_category"].value_counts())

# Outlier status
print("\n6. Outlier Status:")
print(crypto_df["outlier_status"].value_counts())

# Check engineered features
print("\n7. Missing Engineered Features:")
print(
    crypto_df[
        [
            "market_cap_category",
            "volatility_score",
            "supply_ratio"
        ]
    ].isnull().sum()
)

print("\n============================================================")
print("CRYPTOCURRENCY VALIDATION COMPLETED")
print("============================================================")

In [ ]:
# STEP 10.2: SAVE FINAL CLEANED CRYPTOCURRENCY DATASET

crypto_output_file = "final_cleaned_crypto_data.csv"
crypto_df.to_csv(crypto_output_file,index=False)

print("Final cleaned cryptocurrency dataset saved successfully!")
print("File name:", crypto_output_file)
print("Rows:", crypto_df.shape[0])
print("Columns:", crypto_df.shape[1])

In [ ]:
# STEP 10.3: SAVE FINAL CLEANED BOOKS DATASET

books_output_file = "final_cleaned_books_data.csv"
books_df.to_csv(books_output_file,index=False)

print("Final cleaned books dataset saved successfully!")
print("File name:", books_output_file)
print("Rows:", books_df.shape[0])
print("Columns:", books_df.shape[1])

In [ ]:
# STEP 10.4: FINAL DATA QUALITY CHECK

print("============================================================")
print("FINAL DATA QUALITY CHECK")
print("============================================================")

# -------------------- CRYPTO DATA --------------------

print("\nCRYPTOCURRENCY DATASET")
print("----------------------")

print("Rows:", crypto_df.shape[0])
print("Columns:", crypto_df.shape[1])
print("Duplicate IDs:", crypto_df["id"].duplicated().sum())
print("Missing Values:", crypto_df.isnull().sum().sum())
print("Invalid Prices:",((crypto_df["current_price"] <= 0) |(crypto_df["current_price"].isna())).sum())

# -------------------- BOOK DATA --------------------

print("\nBOOK DATASET")
print("------------")

print("Rows:", books_df.shape[0])
print("Columns:", books_df.shape[1])
print("Duplicate Titles:", books_df["title"].duplicated().sum())
print("Missing Values:", books_df.isnull().sum().sum())
print("Invalid Prices:", (books_df["price"] <= 0).sum())
print("Invalid Ratings:",(~books_df["rating"].isin([1, 2, 3, 4, 5])).sum())

print("\n============================================================")
print("FINAL DATA QUALITY CHECK COMPLETED")
print("============================================================")

In [ ]:
# STEP 10.5: REUSABLE CRYPTOCURRENCY CLEANING FUNCTION

def clean_crypto_data(df):
    """
    Cleans and prepares cryptocurrency data
    collected from the CoinGecko API.
    """

    # Remove complex ROI column if present
    if "roi" in df.columns:
        df.drop(columns=["roi"], inplace=True)

    # Remove duplicate cryptocurrencies
    if "id" in df.columns:
        df.drop_duplicates(
            subset="id",
            keep="first",
            inplace=True
        )

    # Convert numeric columns
    numeric_columns = [
        "current_price",
        "market_cap",
        "total_volume",
        "high_24h",
        "low_24h",
        "price_change_24h",
        "price_change_percentage_24h",
        "market_cap_change_24h",
        "market_cap_change_percentage_24h",
        "circulating_supply",
        "total_supply",
        "max_supply"
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # Convert last_updated to datetime
    if "last_updated" in df.columns:
        df["last_updated"] = pd.to_datetime(
            df["last_updated"],
            errors="coerce",
            utc=True
        )

    # Fill important numeric missing values with median
    median_columns = [
        "current_price",
        "market_cap",
        "total_volume",
        "high_24h",
        "low_24h",
        "circulating_supply",
        "total_supply"
    ]

    for column in median_columns:
        if column in df.columns:
            df[column] = df[column].fillna(
                df[column].median()
            )

    # Fill missing percentage-change values with 0
    percentage_columns = [
        "price_change_percentage_24h",
        "market_cap_change_percentage_24h"
    ]

    for column in percentage_columns:
        if column in df.columns:
            df[column] = df[column].fillna(0)

    # Fill remaining change-value missing data with median
    if "price_change_24h" in df.columns:
        df["price_change_24h"] = df["price_change_24h"].fillna(
            df["price_change_24h"].median()
        )

    if "market_cap_change_24h" in df.columns:
        df["market_cap_change_24h"] = df[
            "market_cap_change_24h"
        ].fillna(
            df["market_cap_change_24h"].median()
        )

    # Remove invalid or zero prices
    if "current_price" in df.columns:
        df = df[
            (df["current_price"] > 0) &
            (df["current_price"].notna())
        ].reset_index(drop=True)

    # Market Cap Category
    if "market_cap" in df.columns:
        df["market_cap_category"] = np.select(
            [
                df["market_cap"] >= 10_000_000_000,
                df["market_cap"] >= 1_000_000_000
            ],
            [
                "Large Cap",
                "Mid Cap"
            ],
            default="Small Cap"
        )

    # Volatility Score
    if "price_change_percentage_24h" in df.columns:
        df["volatility_score"] = (
            df["price_change_percentage_24h"].abs()
        )

    # Supply Ratio
    if (
        "circulating_supply" in df.columns and
        "total_supply" in df.columns
    ):
        df["supply_ratio"] = np.where(
            (df["total_supply"].notna()) &
            (df["total_supply"] > 0),
            df["circulating_supply"] /
            df["total_supply"],
            np.nan
        )

    return df


print("Reusable cryptocurrency cleaning function created successfully!")

In [ ]:
# STEP 10.6: REUSABLE BOOKS CLEANING FUNCTION

def clean_books_data(df):
    """
    Cleans and prepares book data
    collected from Books to Scrape.
    """

    # Remove duplicate book titles
    if "title" in df.columns:
        df.drop_duplicates(
            subset="title",
            keep="first",
            inplace=True
        )

        df.reset_index(drop=True, inplace=True)

    # Clean price
    if "price" in df.columns:
        df["price"] = (
            df["price"]
            .astype(str)
            .str.replace("Â£", "", regex=False)
            .str.replace("£", "", regex=False)
            .str.strip()
        )

        df["price"] = pd.to_numeric(
            df["price"],
            errors="coerce"
        )

    # Convert rating words to numbers
    if "rating" in df.columns:
        rating_mapping = {
            "One": 1,
            "Two": 2,
            "Three": 3,
            "Four": 4,
            "Five": 5
        }

        df["rating"] = df["rating"].replace(rating_mapping)

        df["rating"] = pd.to_numeric(
            df["rating"],
            errors="coerce"
        )

    # Clean availability
    if "availability" in df.columns:
        df["availability"] = (
            df["availability"]
            .astype(str)
            .str.strip()
        )

        df["stock_status"] = np.where(
            df["availability"].str.lower() == "in stock",
            "In Stock",
            "Out of Stock"
        )

    # Clean book titles
    if "title" in df.columns:
        df["title"] = (
            df["title"]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    # Create price range category
    if "price" in df.columns:
        df["price_range_category"] = np.select(
            [
                df["price"] < 20,
                df["price"] <= 40
            ],
            [
                "Low",
                "Medium"
            ],
            default="High"
        )

    # Create rating group
    if "rating" in df.columns:
        df["rating_group"] = np.select(
            [
                df["rating"] <= 2,
                df["rating"] == 3
            ],
            [
                "Low Rating",
                "Average Rating"
            ],
            default="High Rating"
        )

    return df


print("Reusable books cleaning function created successfully!")

In [ ]:
# STEP 10.7: FINAL PROJECT DATASET SUMMARY

print("============================================================")
print("FINAL PROJECT DATASET SUMMARY")
print("============================================================")

print("\nCRYPTOCURRENCY DATASET")
print("----------------------")
print("Rows:", crypto_df.shape[0])
print("Columns:", crypto_df.shape[1])
print("Duplicate IDs:", crypto_df["id"].duplicated().sum())
print("Missing Values:", crypto_df.isnull().sum().sum())

print("\nBOOK DATASET")
print("------------")
print("Rows:", books_df.shape[0])
print("Columns:", books_df.shape[1])
print("Duplicate Titles:", books_df["title"].duplicated().sum())
print("Missing Values:", books_df.isnull().sum().sum())

print("\n============================================================")
print("DATA PIPELINE SUMMARY")
print("============================================================")
print("1. CoinGecko API data collection       : Completed")
print("2. Books web scraping                  : Completed")
print("3. Data cleaning                       : Completed")
print("4. Feature engineering                 : Completed")
print("5. Exploratory Data Analysis           : Completed")
print("6. Statistical analysis                : Completed")
print("7. Interactive visualizations          : Completed")
print("8. Outlier detection                   : Completed")
print("9. Final CSV export                    : Completed")
print("10. Reusable cleaning functions        : Completed")

In [ ]:
# STEP 10.8: FINAL PROJECT INSIGHTS

print("FINAL PROJECT INSIGHTS")

# -------------------- BOOK INSIGHTS --------------------

print("\nBOOK DATASET INSIGHTS")
print("---------------------")

highest_avg_price_category = (
    books_df.groupby("category")["price"]
    .mean()
    .idxmax()
)

highest_avg_price = (
    books_df.groupby("category")["price"]
    .mean()
    .max()
)

most_common_rating = books_df["rating"].mode()[0]

average_book_price = books_df["price"].mean()

print(
    f"1. Highest average-priced category: "
    f"{highest_avg_price_category}"
)

print(
    f"   Average price: £{highest_avg_price:.2f}"
)

print(
    f"2. Most common book rating: "
    f"{most_common_rating}/5"
)

print(
    f"3. Overall average book price: "
    f"£{average_book_price:.2f}"
)

print(
    f"4. Rating vs Price Pearson correlation: "
    f"{books_df['rating'].corr(books_df['price']):.4f}"
)

print(
    f"5. Books classified as price outliers: "
    f"{(books_df['price_outlier'] == 'Outlier').sum()}"
)


# -------------------- CRYPTO INSIGHTS --------------------

print("\nCRYPTOCURRENCY DATASET INSIGHTS")
print("--------------------------------")

most_volatile_crypto = (
    crypto_df.loc[
        crypto_df["volatility_score"].idxmax(),
        "name"
    ]
)

highest_volatility = crypto_df["volatility_score"].max()

largest_crypto = (
    crypto_df.loc[
        crypto_df["market_cap"].idxmax(),
        "name"
    ]
)

largest_market_cap = crypto_df["market_cap"].max()

most_common_market_cap = (
    crypto_df["market_cap_category"].mode()[0]
)

print(
    f"1. Most volatile cryptocurrency: "
    f"{most_volatile_crypto}"
)

print(
    f"   Volatility score: "
    f"{highest_volatility:.2f}%"
)

print(
    f"2. Largest cryptocurrency by market capitalization: "
    f"{largest_crypto}"
)

print(
    f"   Market capitalization: "
    f"${largest_market_cap:,.2f}"
)

print(
    f"3. Most common market-cap category: "
    f"{most_common_market_cap}"
)

print(
    f"4. Cryptocurrencies flagged as potential outliers: "
    f"{(crypto_df['outlier_status'] == 'Outlier').sum()}"
)

print(
    f"5. Kruskal-Wallis p-value for volatility differences: "
    f"{kruskal_p:.4f}"
)

if kruskal_p < 0.05:
    print(
        "   Result: Volatility differs significantly "
        "across market-cap categories."
    )
else:
    print(
        "   Result: No statistically significant difference "
        "in volatility across market-cap categories."
    )

In [ ]:
# STEP 10.9: FINAL PROJECT CONCLUSION

print("FINAL PROJECT CONCLUSION")

print("""
This project successfully implemented an advanced multi-source
data pipeline using CoinGecko API data and Books to Scrape web
data.

The pipeline included data collection, data cleaning, feature
engineering, exploratory data analysis, statistical analysis,
outlier detection, interactive visualization, and final dataset
export.

For cryptocurrency data, market capitalization categories,
volatility scores, and supply ratios were created. IQR and
Isolation Forest methods were used for outlier analysis while
legitimate cryptocurrency observations were retained.

For book data, prices and ratings were converted into numerical
formats, duplicate titles were removed, stock status was derived,
and price-range and rating-group features were created.

The analysis showed that book price and rating have a very weak
relationship. The Spearman correlation was approximately 0.03
and the relationship was not statistically significant.

For cryptocurrency volatility, the Kruskal-Wallis test did not
find a statistically significant difference among Large Cap,
Mid Cap, and Small Cap cryptocurrencies.

Overall, the project demonstrates how data from different sources
can be collected, cleaned, transformed, analyzed, visualized,
and converted into reusable datasets and analytical insights.
""")

In [69]:
# Save the final cleaned cryptocurrency dataset

crypto_output_file = "final_cleaned_crypto_data.csv"
crypto_df.to_csv(crypto_output_file,index=False)
print("Crypto CSV saved successfully!")
print("File:", crypto_output_file)

Crypto CSV saved successfully!
File: final_cleaned_crypto_data.csv


In [68]:
# Save the final cleaned books dataset

books_output_file = "final_cleaned_books_data.csv"
books_df.to_csv(books_output_file,index=False)
print("Books CSV saved successfully!")
print("File:", books_output_file)

Books CSV saved successfully!
File: final_cleaned_books_data.csv
